# 绘图## 简介本教程介绍了 **skrf** 的绘图功能。如果您想使用 skrf 的 matplotlib 接口并应用 skrf 的样式，请从这里开始。

In [ ]:
%matplotlib inline

In [ ]:
from matplotlib import pyplot as plt  # for advanced smith chart only

import skrf as rf

## 绘图方法绘图函数被实现为 `Network` 类的成员方法。* `Network.plot_s_re`* `Network.plot_s_im`* `Network.plot_s_mag`* `Network.plot_s_db`* ...类似的方法也存在于阻抗 (`Network.z`) 和导纳参数 (`Network.y`) 中：* `Network.plot_z_re`* `Network.plot_z_im`* ...* `Network.plot_y_re`* `Network.plot_y_im`* ...## 史密斯图作为第一个示例，加载一个 [Network](../api/network.rst)，并在史密斯图上绘制所有四个散射参数。

In [ ]:
from skrf import Network

ring_slot = Network('data/ring slot.s2p')
ring_slot.plot_s_smith()

scikit-rf 包含一个方便的命令，可以快速生成更美观的图形：

In [ ]:
rf.stylely()  # nicer looking. Can be configured with different styles
ring_slot.plot_s_smith()

In [ ]:
ring_slot.plot_s_smith(draw_labels=True)

另一种常见的选择是绘制导纳等值线，而不是阻抗。这可以通过 `chart_type` 参数进行控制。

In [ ]:
ring_slot.plot_s_smith(chart_type='y')

### 带有标记的高级史密斯图史密斯图是一种方便的图表，它显示了复数，其实部和虚部从零到无穷大，并且在中心区域有一个实部参考线，可以进行缩放。但是，由于没有频率轴，因此很难确定哪个点与哪个频率相关。一种常见的解决方法是提供与表格相关的一些频率标记。

In [ ]:
# prepare markers
lines = [
    {'marker_idx': [30, 60, 90], 'color': 'g', 'm': 0, 'n': 0, 'ntw': ring_slot},
    {'marker_idx': [15, 45, 75], 'color': 'r', 'm': 1, 'n': 0, 'ntw': ring_slot},
]

# prepare figure
fig, ax = plt.subplots(1, 1, figsize=(7,8))

# impedance smith chart
rf.plotting.smith(ax = ax, draw_labels = True, ref_imm = 50.0, chart_type = 'z')

# plot data
col_labels = ['Frequency', 'Real Imag']
row_labels = []
row_colors = []
cell_text = []
for l in lines:
    m = l['m']
    n = l['n']
    l['ntw'].plot_s_smith(m=m, n=n, ax = ax, color=l['color'])
    #plot markers
    for i, k in enumerate(l['marker_idx']):
        x = l['ntw'].s.real[k, m, n]
        y = l['ntw'].s.imag[k, m, n]
        z = l['ntw'].z[k, m, n]
        z = f'{z.real:.4f} + {z.imag:.4f}j ohm'
        f = l['ntw'].frequency.f_scaled[k]
        f_unit = l['ntw'].frequency.unit
        row_labels.append(f'M{i + 1}')
        row_colors.append(l['color'])
        ax.scatter(x, y, marker = 'v', s=20, color=l['color'])
        ax.annotate(row_labels[-1], (x, y), xytext=(-7, 7), textcoords='offset points', color=l['color'])
        cell_text.append([f'{f:.3f} {f_unit}', z])
leg1 = ax.legend(loc="upper right", fontsize= 6)

# plot the table
the_table = ax.table(cellText=cell_text,
                      colWidths=[0.4] * 2,
                      rowLabels=row_labels,
                      colLabels=col_labels,
                      rowColours=row_colors,
                      loc='bottom')
the_table.auto_set_font_size(False)
the_table.set_fontsize(6)
#the_table.scale(1.5, 1.5)

### 带背景的高级史密斯图可以在高分辨率史密斯图背景之上绘制史密斯图（或者根据您的想象绘制其他图像）。

In [ ]:
# prepare figure
fig, ax = plt.subplots(1, 1, figsize=(8,8))
background = plt.imread('figures/smithchart.png')

# tweak background position
ax.imshow(background, extent=[-1.185, 1.14, -1.13, 1.155])
rf.plotting.smith(ax = ax, draw_labels = True, ref_imm = 1.0, chart_type = 'z')

ring_slot.plot_s_smith(ax = ax)

有关如何自定义史密斯图的更多信息，请参阅 `skrf.plotting.smith()`。## 复数平面网络参数也可以在复数平面中绘制，而无需使用史密斯图，方法是使用 `Network.plot_s_complex`。

In [ ]:
from matplotlib import pyplot as plt

ring_slot.plot_s_complex()

plt.axis('equal') # otherwise circles won't be circles

## 对数幅度复数网络参数的标量分量也可以与频率一起绘制。要绘制散射参数的对数幅度与频率的关系图，

In [ ]:
ring_slot.plot_s_db()

当绘图方法不带任何参数时，将绘制所有参数。可以通过传递索引 `m` 和 `n` 到绘图命令来绘制单个参数（索引从 0 开始）。将环形槽的模拟反射系数与测量值进行比较。

In [ ]:
from skrf.data import ring_slot_meas

ring_slot.plot_s_db(m=0,n=0, label='Theory')
ring_slot_meas.plot_s_db(m=0,n=0, label='Measurement')

## 相位绘制相位图，

In [ ]:
ring_slot.plot_s_deg()

或者展开后的相位。

In [ ]:
ring_slot.plot_s_deg_unwrap()

相位以弧度（rad）为单位，也可以使用。

## 群时延

`Network` 对象有一个 `plot()` 方法，它可以创建一个矩形图，将参数与频率进行对比。这可以用来创建自定义的图表，而不是使用预定义的图表。例如，群时延。

In [ ]:
gd = abs(ring_slot.s21.group_delay) *1e9 # in ns

ring_slot.plot(gd)
plt.ylabel('Group Delay (ns)')
plt.title('Group Delay of Ring Slot S21')

## 阻抗、导纳阻抗和导纳参数的各个分量可以以类似的方式进行绘制。

In [ ]:
ring_slot.plot_z_im()

In [ ]:
ring_slot.plot_y_im()

## 自定义绘图图例条目会自动填充为 `Network.name`。可以通过将 `label` 参数传递给绘图方法来覆盖该条目。

In [ ]:
ring_slot.plot_s_db(m=0,n=0, label = 'Simulation')

x 轴上使用的频率单位将自动从 `Network.frequency.unit` 属性中填充。要更改标签，请更改频率的 `unit`。

In [ ]:
ring_slot.frequency.unit = 'mhz'
ring_slot.plot_s_db(0,0)

传递给绘图方法的其他关键字参数会传递给 matplotlib 的 `matplotlib.pyplot.plot` 函数。

In [ ]:
ring_slot.frequency.unit='ghz'
ring_slot.plot_s_db(m=0,n=0, linewidth = 3, linestyle = '--', label = 'Simulation')
ring_slot_meas.plot_s_db(m=0,n=0, marker = 'o', markevery = 10,label = 'Measured')


可以通过 [matplotlib](http://matplotlib.sourceforge.net) 函数自定义所有绘图组件，并且可以使用上下文管理器来使用 `styles`。

In [ ]:
from matplotlib import pyplot as plt
from matplotlib import style

mpl_style = "seaborn-ticks"
mpl_style = mpl_style if mpl_style in style.available else "seaborn-v0_8-ticks"

with style.context(mpl_style):
    ring_slot.plot_s_smith()
    plt.xlabel('Real Part')
    plt.ylabel('Imaginary Part')
    plt.title('Smith Chart With Legend Room')
    plt.axis([-1.1,2.1,-1.1,1.1])
    plt.legend(loc=5)


## 保存图形可以使用 matplotlib 提供的 GUI 将图形保存为各种文件格式。但是，skrf 提供了一个便捷函数，名为 `skrf.plotting.save_all_figs`，它允许将所有打开的图形以多种文件格式保存到磁盘，文件名将从每个图形的标题中提取。

In [ ]:
from skrf.plotting import save_all_figs

save_all_figs('data/', format=['png','eps','pdf'])

## 在绘图后添加标记一个常见的需求是生成一个彩色图，以便在灰度打印中也能清晰显示。`skrf.plotting.add_markers_to_lines` 函数会在图已经绘制完成后，为图中的每一条线添加不同的标记，通常是在你意识到需要添加标记时才使用的。

In [ ]:
from skrf import plotting

with plt.style.context('grayscale'):
    ring_slot.plot_s_deg()
    plotting.add_markers_to_lines()
    plt.legend() # have to re-generate legend
